# Stage II (lesions) - 05. Overlap of the segmentation splits with the grading test pool

FGADR, IDRiD and MAPLES-DR (derived from Messidor) contribute both lesion masks (Stage II) and
graded images (Stages I and III). The lesion pools were split independently of the grading
split, so some grading-test images may have been used to train a lesion segmenter.

1. **Configuration** - grading test list and the four lesion manifests.
2. **Re-created splits** - every lesion split is re-created exactly (seed 42) and asserted to
   reproduce the paper sizes.
3. **Overlap count** - file names are normalised (folder, extension, case and lesion suffixes
   removed) and matched against the grading-test images of FGADR, IDRiD and Messidor-2.
   Optionally, the whole vessel pool is counted as well (an upper bound, because its split
   cannot be replayed).

IDRiD segmentation images are numbered differently from the IDRiD grading images, so a name
match cannot rule out overlap there. Paper run: 259 grading-test images (239 of 277 FGADR and
20 of 262 Messidor-2) were in the training split of at least one lesion model, and 27
Messidor-2 test images belong to the vessel pool.

## Step 1. Configuration

In [ ]:
from pathlib import Path

TEST_CSV = Path("data/test/test_grade.csv")         # grading test pool, column "image"
DATA_ROOT = Path("data/stage2_lesions")
MANIFESTS = {les: DATA_ROOT / les / "manifest.csv" for les in ("MA", "HE", "EX", "CWS")}
VESSEL_IMG_DIR = Path("data/vessel_dataset/image")     # set to None to skip the vessel pool
OUT_CSV = Path("outputs/stage2_lesion_eval/stage2_overlap.csv")

## Step 2-3. Re-created splits and overlap count

In [ ]:
from hierarchiretina.stage2.lesion_data import stage2_overlap

table, train_hits, n_vessel = stage2_overlap(TEST_CSV, MANIFESTS, VESSEL_IMG_DIR)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
table.to_csv(OUT_CSV, index=False)
table.pivot_table(index=["model", "split"], columns="grading_test_source", values="n_overlap",
                  aggfunc="sum").fillna(0).astype(int)

In [ ]:
print(f"distinct grading-test images in any lesion TRAIN split: {len(train_hits)} "
      f"{train_hits.src.value_counts().to_dict()}")
if n_vessel is not None:
    print(f"distinct grading-test images in the vessel pool: {n_vessel}")